# Montreal playground MVP data prep

This notebook prepares the data for the A2 map. It reads the source GeoJSON, filters the features to `TYPE == "Aire de jeu - enfant"`, validates the result, and writes a smaller GeoJSON file for the map.


In [1]:
import json
from pathlib import Path

source_path = Path("../data/terrain_sport_ext.json")
output_path = Path("../data/playgrounds.geojson")

print("Source:", source_path)
print("Output:", output_path)


Source: ..\data\terrain_sport_ext.json
Output: ..\data\playgrounds.geojson


In [2]:
with source_path.open("r", encoding="utf-8") as f:
    data = json.load(f)

features = data.get("features", [])
print("Total features:", len(features))
print("Sample keys:", sorted(features[0]["properties"].keys()))
print("Sample TYPE:", features[0]["properties"].get("TYPE"))
print("Sample ARROND:", features[0]["properties"].get("ARROND"))


Total features: 3476
Sample keys: ['ARROND', 'ID', 'INDEX_PARC', 'NOM', 'OBJECTID', 'TYPE']
Sample TYPE: Aire de jeu - enfant
Sample ARROND: Le Plateau-Mont-Royal


In [3]:
playgrounds = {
    "type": "FeatureCollection",
    "features": [
        feature
        for feature in features
        if feature.get("properties", {}).get("TYPE") == "Aire de jeu - enfant"
    ]
}

removed_count = len(features) - len(playgrounds["features"])
print("Features removed:", removed_count)

output_path.write_text(json.dumps(playgrounds, ensure_ascii=False), encoding="utf-8")
print("Playground features written:", len(playgrounds["features"]))


Features removed: 2468
Playground features written: 1008


## Validate the output

These checks confirm the file is ready for the map and matches the project requirement.


In [4]:
with output_path.open("r", encoding="utf-8") as f:
    result = json.load(f)

result_features = result["features"]

assert len(result_features) == 1008, f"Expected 1008 features, found {len(result_features)}"
assert all(feature.get("properties", {}).get("TYPE") == "Aire de jeu - enfant" for feature in result_features)
assert all(feature.get("geometry") and feature["geometry"].get("coordinates") is not None for feature in result_features)
assert all(feature.get("properties", {}).get("ARROND") not in (None, "") for feature in result_features)
assert all(
    -74.5 <= feature["geometry"]["coordinates"][0] <= -73.4 and 45.3 <= feature["geometry"]["coordinates"][1] <= 46.0
    for feature in result_features
)

print("Validated features:", len(result_features))
print("All checks passed")


Validated features: 1008
All checks passed
